# Lecture 6: NLTK Part 2 — POS Tagging and Corpus Comparison

Last time we covered accessing NLTK corpora, frequency distributions, conditional frequency distributions, text objects, and bigrams. Today we'll build on those tools to explore **Part-of-Speech (POS) tagging** and use everything together to compare corpora.

**Prerequisites:**
- Python and the `ling250` environment
- Familiarity with `FreqDist`, `ConditionalFreqDist`, and `bigrams` from Lecture 5
- `Night_Vale.txt` in your `data/` folder

---

## Part 1: Bigrams Recap

Quick reminder from last time: **bigrams** are pairs of consecutive items. We used them with `ConditionalFreqDist` to build a simple text generator.

In [ ]:
import nltk
from nltk.util import bigrams
from nltk import FreqDist, ConditionalFreqDist
from nltk.corpus import gutenberg

# Bigrams are pairs of consecutive items
example = ['the', 'cat', 'sat', 'on', 'the', 'mat']
print(list(bigrams(example)))

In [ ]:
import random

# We used bigrams + CFD to generate text
emma_words = gutenberg.words('austen-emma.txt')
emma_cfd = ConditionalFreqDist(bigrams(emma_words))

# Generate 20 words starting from "Emma"
current_word = 'Emma'
generated = [current_word]

for i in range(20):
    following_words = list(emma_cfd[current_word].keys())
    frequencies = list(emma_cfd[current_word].values())
    if not following_words:
        break
    current_word = random.choices(following_words, weights=frequencies)[0]
    generated.append(current_word)

print(' '.join(generated))

Today we'll apply the same idea — bigrams and frequency distributions — but to **Part-of-Speech tags** instead of words.

---

## Part 2: Part-of-Speech Tagging

### Quick review: What are Parts of Speech?

Parts of Speech (POS) are grammatical categories for words:
- **Nouns** (dog, city, idea)
- **Verbs** (run, think, is)
- **Adjectives** (big, red, interesting)
- **Adverbs** (quickly, very, often)
- **Determiners** (the, a, this)
- **Prepositions** (in, on, with)
- etc.

POS tagging is the process of automatically assigning a tag to each word in a text. NLTK has a built-in tagger for English.

### Using `nltk.pos_tag()`

`pos_tag()` takes a **list of words** (tokens) and returns a list of `(word, tag)` tuples.

In [ ]:
# You may need to run this the first time:
# nltk.download('averaged_perceptron_tagger_eng')

sentence = ['The', 'dog', 'chased', 'the', 'cat', 'quickly']
tagged = nltk.pos_tag(sentence)
print(tagged)

The tags like `DT`, `NN`, `VBD`, `RB` come from the **Penn Treebank tagset**. There are dozens of fine-grained tags. Some common ones:

| Tag | Meaning | Example |
|-----|---------|--------|
| NN | Noun, singular | dog |
| NNS | Noun, plural | dogs |
| NNP | Proper noun | Emma |
| VB | Verb, base form | run |
| VBD | Verb, past tense | ran |
| VBG | Verb, gerund | running |
| JJ | Adjective | big |
| RB | Adverb | quickly |
| DT | Determiner | the |
| IN | Preposition | in, on |
| PRP | Personal pronoun | she, he |

In [ ]:
# Try a more interesting sentence
sentence2 = ['I', 'saw', 'the', 'man', 'with', 'the', 'telescope']
nltk.pos_tag(sentence2)

### The Universal Tagset

The Penn Treebank tags are very detailed, which can be overwhelming. NLTK can map them to a simpler **universal tagset** with just 12 categories:

In [ ]:
# nltk.download('universal_tagset')  # uncomment if needed

nltk.pos_tag(sentence, tagset='universal')

| Universal Tag | Meaning |
|---------------|--------|
| NOUN | Noun |
| VERB | Verb |
| ADJ | Adjective |
| ADV | Adverb |
| DET | Determiner |
| ADP | Adposition (preposition) |
| PRON | Pronoun |
| CONJ | Conjunction |
| NUM | Numeral |
| PRT | Particle |
| X | Other |
| . | Punctuation |

In [ ]:
# Tag a longer passage from Emma
emma_sents = gutenberg.sents('austen-emma.txt')
tagged_sent = nltk.pos_tag(emma_sents[0])
print(tagged_sent)

**Important:** `nltk.pos_tag()` only works for **English**. For other languages, you'd need a language-specific tagger.

---

## Part 3: Tagged Corpora

Some NLTK corpora come **pre-tagged** by human annotators. This is more reliable than automatic tagging and lets us study how tags are actually distributed in real text.

### The Brown Corpus (tagged)

The Brown Corpus was one of the first electronically tagged corpora. We already used `brown.words()` and `brown.sents()` — now we'll use the tagged versions.

In [ ]:
from nltk.corpus import brown

# tagged_words() returns (word, tag) tuples
brown.tagged_words()[:20]

In [ ]:
# tagged_sents() returns lists of (word, tag) tuples
brown.tagged_sents()[0]

In [ ]:
# We can also use the universal tagset with tagged corpora
brown.tagged_words(tagset='universal')[:20]

In [ ]:
# Filter by category, just like with words()
brown.tagged_words(categories='news', tagset='universal')[:20]

### The Treebank Corpus

The Penn Treebank is another classic tagged corpus, widely used in NLP research.

In [ ]:
from nltk.corpus import treebank

# nltk.download('treebank')  # uncomment if needed

treebank.tagged_words()[:20]

**Key point:** `tagged_words()` and `tagged_sents()` work exactly like `words()` and `sents()`, but each word comes paired with its POS tag.

---

## Part 4: Analyzing Tags with FreqDist and CFD

Since tagged words are just `(word, tag)` tuples, we can use the same tools from last time — `FreqDist` and `ConditionalFreqDist` — to analyze POS patterns.

### What are the most common tags?

We can extract just the tags from a tagged corpus and count them.

In [ ]:
# Extract tags from the Brown corpus (universal tagset)
brown_tags = [tag for (word, tag) in brown.tagged_words(tagset='universal')]

tag_fdist = FreqDist(brown_tags)
tag_fdist.most_common()

In [ ]:
tag_fdist.plot()

**Observation:** Nouns and verbs dominate, followed by determiners and prepositions. This tells us something fundamental about the structure of English text.

### ConditionalFreqDist: Most common words per tag

We can use a CFD to answer: "For each POS tag, what are the most common words?"

In [ ]:
# Condition = tag, Sample = word
tag_word_pairs = [
    (tag, word.lower())
    for (word, tag) in brown.tagged_words(tagset='universal')
]

tag_word_cfd = ConditionalFreqDist(tag_word_pairs)

In [ ]:
# What are the most common nouns?
tag_word_cfd['NOUN'].most_common(15)

In [ ]:
# Most common verbs?
tag_word_cfd['VERB'].most_common(15)

In [ ]:
# Most common adjectives?
tag_word_cfd['ADJ'].most_common(15)

### CFD: Most common tags per genre

A more interesting question: do different genres of text use parts of speech differently?

In [ ]:
# Condition = genre, Sample = tag
genres = ['news', 'romance', 'science_fiction', 'humor', 'religion']

genre_tag_pairs = [
    (genre, tag)
    for genre in genres
    for (word, tag) in brown.tagged_words(categories=genre, tagset='universal')
]

genre_tag_cfd = ConditionalFreqDist(genre_tag_pairs)

In [ ]:
# Compare tag distributions across genres
genre_tag_cfd.tabulate(
    samples=['NOUN', 'VERB', 'ADJ', 'ADV', 'PRON', 'DET']
)

In [ ]:
genre_tag_cfd.plot(
    samples=['NOUN', 'VERB', 'ADJ', 'ADV', 'PRON', 'DET']
)

**Discussion:** What patterns do you notice? Which genres use more pronouns? More adjectives? Why might that be?

### Tag Bigrams

Just as we made bigrams of words, we can make bigrams of **tags**. This tells us about common grammatical sequences — e.g., determiners are usually followed by nouns or adjectives.

In [ ]:
# Get all tags from the Brown corpus
brown_tags = [
    tag
    for (word, tag) in brown.tagged_words(tagset='universal')
]

# Make bigrams of tags
tag_bigrams = list(bigrams(brown_tags))
print(tag_bigrams[:10])

In [ ]:
# Build a CFD from tag bigrams
tag_bigram_cfd = ConditionalFreqDist(tag_bigrams)

In [ ]:
# What tags most commonly follow a determiner (DET)?
tag_bigram_cfd['DET'].most_common(5)

In [ ]:
# What tags most commonly follow a noun?
tag_bigram_cfd['NOUN'].most_common(5)

In [ ]:
# Tabulate the full picture
main_tags = ['DET', 'NOUN', 'VERB', 'ADJ', 'ADV', 'ADP', 'PRON']
tag_bigram_cfd.tabulate(
    conditions=main_tags,
    samples=main_tags
)

**Reading this table:** Each row is the *first* tag in a bigram, each column is the *second* tag. So the cell at (DET, NOUN) tells you how often a determiner is followed by a noun.

**Discussion:** What patterns do you see? Do they match your intuitions about English grammar?

---

## Part 5: Practical Example — Night Vale vs. Brown Corpus

Now let's put everything together. How would we start to **quantify the differences** between Night Vale and a section of the Brown Corpus?

This is the kind of question that comes up in corpus linguistics all the time: given two texts or collections of text, what are the measurable differences between them?

### Setup: Load and tag Night Vale

In [ ]:
from nltk.corpus import PlaintextCorpusReader

nv_corpus = PlaintextCorpusReader('../data/', 'Night_Vale.txt')
nv_words = list(nv_corpus.words('Night_Vale.txt'))
print(f"Night Vale: {len(nv_words)} words")

# We'll compare with the 'fiction' category of Brown
fiction_words = list(brown.words(categories='fiction'))
print(f"Brown fiction: {len(fiction_words)} words")

In [ ]:
# Tag Night Vale (this might take a moment)
nv_tagged = nltk.pos_tag(nv_words, tagset='universal')
print(nv_tagged[:15])

In [ ]:
# Get tagged fiction words from Brown (these are already tagged)
fiction_tagged = brown.tagged_words(categories='fiction', tagset='universal')
print(list(fiction_tagged)[:15])

### Approach 1: Compare tag frequencies

Are the distributions of POS tags different? Since the corpora are different sizes, we need to **normalize** counts to proportions (frequencies).

In [ ]:
# Count tags in each corpus
nv_tag_fdist = FreqDist(tag for (word, tag) in nv_tagged)
fiction_tag_fdist = FreqDist(tag for (word, tag) in fiction_tagged)

# Get total counts for normalization
nv_total = sum(nv_tag_fdist.values())
fiction_total = sum(fiction_tag_fdist.values())

print(f"Night Vale total tagged tokens: {nv_total}")
print(f"Fiction total tagged tokens: {fiction_total}")

In [ ]:
# Compare proportions for main tags
main_tags = ['NOUN', 'VERB', 'ADJ', 'ADV', 'PRON', 'DET', 'ADP']

print(f"{'Tag':<8} {'Night Vale':>12} {'Brown Fiction':>14}")
print("-" * 36)
for tag in main_tags:
    nv_prop = nv_tag_fdist[tag] / nv_total
    fiction_prop = fiction_tag_fdist[tag] / fiction_total
    print(f"{tag:<8} {nv_prop:>11.3f} {fiction_prop:>13.3f}")

**Discussion:** Do you see any differences? Think about *why* they might exist — what's different about Night Vale as a text compared to typical fiction?

### Approach 2: Compare word usage within a tag

For a specific tag, what are the most common words in each corpus?

In [ ]:
# Most common nouns in each
nv_nouns = FreqDist(
    word.lower() for (word, tag) in nv_tagged if tag == 'NOUN'
)
fiction_nouns = FreqDist(
    word.lower() for (word, tag) in fiction_tagged if tag == 'NOUN'
)

print("Night Vale top nouns:")
print(nv_nouns.most_common(15))
print()
print("Brown fiction top nouns:")
print(fiction_nouns.most_common(15))

### Approach 3: Compare tag bigrams

Do the two corpora differ in their grammatical patterns?

In [ ]:
# Tag bigrams for each corpus
nv_tag_seq = [tag for (word, tag) in nv_tagged]
fiction_tag_seq = [tag for (word, tag) in fiction_tagged]

nv_tag_bigrams = FreqDist(bigrams(nv_tag_seq))
fiction_tag_bigrams = FreqDist(bigrams(fiction_tag_seq))

print("Night Vale most common tag bigrams:")
print(nv_tag_bigrams.most_common(10))
print()
print("Brown fiction most common tag bigrams:")
print(fiction_tag_bigrams.most_common(10))

### Why normalize?

The raw counts above aren't directly comparable because the corpora are different sizes. To compare fairly, divide by the total number of bigrams in each corpus:

In [ ]:
nv_bigram_total = sum(nv_tag_bigrams.values())
fiction_bigram_total = sum(fiction_tag_bigrams.values())

# Compare proportions for the most common bigrams
top_bigrams = [bg for (bg, count) in nv_tag_bigrams.most_common(8)]

print(f"{'Tag Bigram':<18} {'Night Vale':>12} {'Brown Fiction':>14}")
print("-" * 46)
for bg in top_bigrams:
    nv_prop = nv_tag_bigrams[bg] / nv_bigram_total
    fiction_prop = fiction_tag_bigrams[bg] / fiction_bigram_total
    label = f"{bg[0]}->{bg[1]}"
    print(f"{label:<18} {nv_prop:>11.4f} {fiction_prop:>13.4f}")

### Other ideas to explore

There are many more ways to compare corpora. Some ideas:

- **Vocabulary richness:** How many unique words does each corpus use? (Hint: `len(set(words))` vs `len(words)`)
- **Sentence length:** What's the average sentence length in each?
- **Word bigrams:** Are certain word pairs unique to one corpus?
- **Tags for specific words:** Does the same word get tagged differently in different contexts?

---

## Practice Exercises

Try these on your own!

### Exercise 1: Tag a passage

Take the first 3 sentences from Night Vale. POS-tag them using both the default tagset and the universal tagset. Are there any tags that surprise you or seem wrong?

In [ ]:
# Your code here


### Exercise 2: Genre tag profiles

Pick three Brown corpus genres. For each genre, compute the **proportion** of each universal tag (not the raw count). Which genre has the highest proportion of adjectives? Of pronouns?

In [ ]:
# Your code here


### Exercise 3: What follows a verb?

Using tag bigrams in the Brown corpus, what are the 5 most common tags that follow a VERB? Now do the same for Night Vale. Are the patterns similar?

In [ ]:
# Your code here


### Exercise 4: Vocabulary richness

Compute the **type-token ratio** (number of unique words / total words) for Night Vale and at least two Brown corpus genres. Which has the richest vocabulary? Does this make sense given the nature of each text?

In [ ]:
# Your code here


---

## Quick Reference

### POS Tagging

| Code | What it does |
|------|--------------|
| `nltk.pos_tag(words)` | Tag a list of words (Penn Treebank tags) |
| `nltk.pos_tag(words, tagset='universal')` | Tag with simplified universal tags |

### Tagged Corpora

| Method | Returns |
|--------|---------|
| `.tagged_words()` | List of `(word, tag)` tuples |
| `.tagged_sents()` | List of sentences, each a list of `(word, tag)` tuples |
| `.tagged_words(tagset='universal')` | Same, but with universal tags |
| `.tagged_words(categories='news')` | Filter by category |

### Common Downloads

| Download | For |
|----------|---------|
| `nltk.download('averaged_perceptron_tagger_eng')` | `pos_tag()` |
| `nltk.download('universal_tagset')` | `tagset='universal'` |
| `nltk.download('brown')` | Brown corpus |
| `nltk.download('treebank')` | Penn Treebank corpus |

### Universal Tagset Summary

| Tag | Meaning | Tag | Meaning |
|-----|---------|-----|--------|
| NOUN | Noun | PRON | Pronoun |
| VERB | Verb | CONJ | Conjunction |
| ADJ | Adjective | NUM | Numeral |
| ADV | Adverb | PRT | Particle |
| DET | Determiner | X | Other |
| ADP | Adposition | . | Punctuation |

---

## Further Reading

- [NLTK Book Chapter 5](https://www.nltk.org/book/ch05.html) — Categorizing and Tagging Words
- [Universal POS Tags](https://universaldependencies.org/u/pos/) — the Universal Dependencies tagset (related but not identical to NLTK's universal tagset)
- [Penn Treebank Tagset](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html) — full list of Penn Treebank tags